In [253]:
import pickle, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression,Ridge,Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score,mean_squared_error,mean_absolute_error



In [254]:
df = pd.read_csv('../data/processed/cleaned_bangalore_house_data.csv')
df.head()

,total_sqft,bath,balcony,price,bhk,location_1st Block Jayanagar,location_1st Phase JP Nagar,location_2nd Phase Judicial Layout,location_2nd Stage Nagarbhavi,location_5th Phase JP Nagar,...,location_Vishveshwarya Layout,location_Vishwapriya Layout,location_Vittasandra,location_Whitefield,location_Yelachenahalli,location_Yelahanka,location_Yelahanka New Town,location_Yelenahalli,location_Yeshwanthpur,location_other
0,1250.0,2.0,2.0,40.0,2,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1200.0,2.0,2.0,83.0,2,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1170.0,2.0,2.0,40.0,2,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1425.0,2.0,2.0,65.0,3,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,947.0,2.0,2.0,43.0,2,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [255]:
# get X(Feature) and Y(Target) value
X = df.drop(["price"], axis=1)
Y = (df['price'])

In [256]:
# split the data into tain and test
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

In [257]:
X_train.shape

(5510, 241)

In [258]:
X_test.shape

(1378, 241)

In [259]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [260]:
X_train_scaled

array([[-0.7110394 , -0.42998729, -0.69910919, ..., -0.03013744,
        -0.07521946, -0.43005376],
       [-0.48876195, -0.42998729, -0.69910919, ..., -0.03013744,
        -0.07521946, -0.43005376],
       [-0.40632792, -0.42998729,  0.56677698, ..., -0.03013744,
        -0.07521946, -0.43005376],
       ...,
       [-0.52703489, -0.42998729, -0.69910919, ..., -0.03013744,
        -0.07521946, -0.43005376],
       [-0.27531671, -0.42998729, -0.69910919, ..., -0.03013744,
        -0.07521946, -0.43005376],
       [-0.70809533, -0.42998729, -0.69910919, ..., -0.03013744,
        -0.07521946, -0.43005376]], shape=(5510, 241))

In [261]:
models = {
    "LinearRegression" : LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(),
}

In [262]:
parameters = {
    "LinearRegression": {},
    "Ridge":{
        'alpha': [0.01, 0.1, 1, 10, 50, 100, 150, 200, 250, 500]
    },
    "Lasso":{
        'alpha': [0.01, 0.1, 1, 10, 50, 100, 150, 200, 250, 500]
    }
}

In [263]:
result = []
trained_model = {}
for name, model in models.items():
    grid = GridSearchCV(
        estimator=model,
        param_grid=parameters[name],
        cv=5,
        scoring='r2'
    )
    grid.fit(X_train_scaled,y_train)
    
    #save model
    best_model = grid.best_estimator_
    trained_model[name] = best_model

    y_train_pred = best_model.predict(X_train_scaled)
    y_test_pred = best_model.predict(X_test_scaled)
    
    r2_train_value = r2_score(y_train,y_train_pred)
    r2_test_value = r2_score(y_test,y_test_pred)
    
    MAE_value = mean_absolute_error(y_test,y_test_pred)
    MSE_value = mean_squared_error(y_test,y_test_pred)
    
    
    result.append({
        "model": name,
        "best_parameter": grid.best_params_,
        "r2_train": r2_train_value,
        "r2_test": r2_test_value,
        "mae": MAE_value,
        "mse":MSE_value
    })
    
    
    

In [264]:
result_df = pd.DataFrame(result)
result_df = result_df.sort_values(
    by='r2_test',
    ascending=False
)
print(result_df)

              model  best_parameter  r2_train   r2_test        mae  \
1             Ridge  {'alpha': 150}  0.859137  0.789564  16.422678   
2             Lasso  {'alpha': 0.1}  0.859924  0.774612  16.638223   
0  LinearRegression              {}  0.860319  0.774207  16.635513   

           mse  
1  1143.542498  
2  1224.793923  
0  1226.996795  


In [265]:
best_model = result_df.iloc[0]['model']
print(best_model)

Ridge


In [266]:
final_model = trained_model[best_model]
print(final_model)

Ridge(alpha=150)


## Save ML model and colums

In [269]:

with open("../app/models/house_price_pred_model.pkl", "wb") as f:
    pickle.dump(final_model, f)
 
with open("../app/models/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)
   
columns = {
    "data_columns": X.columns.tolist()
}
with open("../app/models/columns.json", "w") as f:
    json.dump(columns, f)